# FRM · View Teacher Labels (`imp_answer` / `imp_question`)

Loads the Phase-1 label cache from `MyDrive/frm_out/labels/` and shows what the
importance labels look like: shapes, a single sample as a 9x9 grid, heatmaps, and
summary stats. Also **exports** `imp_answer.npy` / `imp_question.npy` as standalone
files (they normally live inside `imp.npz`). CPU runtime is fine — no GPU needed.

In [ ]:
# 1) mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) point at the label cache + list what's there
import os, json, numpy as np
LABELS = "/content/drive/MyDrive/frm_out/labels"
print("files in", LABELS, "->")
for f in sorted(os.listdir(LABELS)):
    print("  ", f, f"({os.path.getsize(os.path.join(LABELS,f))/1e6:.1f} MB)")

In [ ]:
# 3) load the importance arrays (from imp.npz; fall back to standalone .npy)
npz = os.path.join(LABELS, "imp.npz")
if os.path.exists(npz):
    z = np.load(npz)
    print("keys in imp.npz:", z.files)
    imp_answer   = z["imp_answer"]
    imp_question = z["imp_question"]
    imp_context  = z["imp_context"] if "imp_context" in z.files else None
    imp_loo      = z["imp_loo"] if "imp_loo" in z.files else None
else:
    imp_answer   = np.load(os.path.join(LABELS, "imp_answer.npy"))
    imp_question = np.load(os.path.join(LABELS, "imp_question.npy"))
    imp_context = imp_loo = None

print("imp_answer  :", imp_answer.shape, imp_answer.dtype)   # [N, 81]
print("imp_question:", imp_question.shape, imp_question.dtype)
print("each row = one sample, each of the 81 columns = one 9x9 global cell")

In [ ]:
# 4) export the two arrays as standalone .npy files (what you asked for)
np.save(os.path.join(LABELS, "imp_answer.npy"),   imp_answer)
np.save(os.path.join(LABELS, "imp_question.npy"), imp_question)
print("wrote imp_answer.npy and imp_question.npy into", LABELS)

In [ ]:
# 5) load the per-sample metadata (id, question, answer, gaze cell)
metas = [json.loads(l) for l in open(os.path.join(LABELS, "meta.jsonl")) if l.strip()]
print("samples:", len(metas))
for m in metas[:5]:
    print(f"row={m['row']:4d}  {m['question_type']:<28} gaze_cell={m['g_idx']:2d}"
          f"  Q: {m['question'][:55]!r} -> A: {m['answer'][:35]!r}")

### Inspect one sample as a 9x9 grid
Change `i` to look at other samples. The label is an 81-vector; reshaping to 9x9
shows *where on the thumbnail the answer's attention landed*.

In [ ]:
GRID = 9
i = 0                      # <-- change to inspect a different sample
m = metas[i]
print("id:", m["id"], "| type:", m["question_type"])
print("Q :", m["question"])
print("A :", m["answer"])
gr, gc = m["g_idx"] // GRID, m["g_idx"] % GRID
print(f"gaze cell g_idx={m['g_idx']}  (row {gr}, col {gc})")

va = imp_answer[i]
print("\nimp_answer as 9x9 (rounded):")
print(np.round(va.reshape(GRID, GRID), 3))
print("\ntop-5 cells by answer importance:", np.argsort(-va)[:5].tolist())
print("raw imp_answer[i][:10]:", np.round(va[:10], 4))

### Heatmaps — `imp_answer` vs `imp_question` (red x = gaze cell)

In [ ]:
import matplotlib.pyplot as plt

def show(i):
    m = metas[i]; gr, gc = m['g_idx'] // GRID, m['g_idx'] % GRID
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
    for a, arr, name in [(ax[0], imp_answer[i], 'imp_answer (label)'),
                         (ax[1], imp_question[i], 'imp_question (control)')]:
        im = a.imshow(arr.reshape(GRID, GRID), cmap='viridis')
        a.scatter([gc], [gr], c='red', marker='x', s=140)
        a.set_title(name); fig.colorbar(im, ax=a, fraction=0.046)
    fig.suptitle(f"[{i}] {m['question_type']} — {m['question'][:70]}", fontsize=10)
    fig.tight_layout(); plt.show()

show(0)

### Summary across all samples
Where does the answer's attention usually land, and how sparse is the signal?

In [ ]:
print("N samples      :", len(imp_answer))
print("imp_answer  min/max/mean: %.4f / %.4f / %.4f" % (imp_answer.min(), imp_answer.max(), imp_answer.mean()))
print("imp_question min/max/mean: %.4f / %.4f / %.4f" % (imp_question.min(), imp_question.max(), imp_question.mean()))

avg = imp_answer.mean(0).reshape(GRID, GRID)
plt.figure(figsize=(4.5, 4))
plt.imshow(avg, cmap="magma"); plt.colorbar(fraction=0.046)
plt.title("mean imp_answer per cell\n(all samples)"); plt.show()

### (optional) Peek at the global tokens `G`
The `[N, 81, d]` float16 memmap — the actual thumbnail tokens each label refers to.

In [ ]:
d = metas[0]["d"]; N = len(metas)
G = np.memmap(os.path.join(LABELS, "G.fp16.dat"), dtype=np.float16, mode="r", shape=(N, 81, d))
print("G shape:", G.shape, "(dtype float16)")
print("G[0], first 3 tokens x first 6 dims:\n", np.round(np.asarray(G[0][:3, :6]), 3))